# Refuerzo y cerebro modular

Votación de decisiones, entornos multicuerpo, biomas neuronales, cerebro modular con expertos y tálamo cortical.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

## Resultados en el banco

RlNeuron, la red de votación, los biomas, el cerebro modular (CorticalMoE) y el tálamo están medidos en `laboratorio/banco/piezas_06.py` contra la red estándar y una mezcla de expertos de la industria (5 semillas, 7 tareas). Resumen: ninguno le gana a la tecnología actual; en la tarea caótica el cerebro modular sube un poco, pero con 12 veces más parámetros, y Dropout lo supera. Detalle y arreglos en `laboratorio/banco/resultados-06.md`.

Quedan para la parte final: el entorno de refuerzo para código, los cubos con varios cuerpos y ReroutingNetwork.

Medido en el banco: tal cual es inestable (según la semilla llega al óptimo o se derrumba); con REINFORCE estándar (recompensa futura menos el promedio) llega al óptimo siempre.

##neuronas individuales con RL

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- Neurona con Bucle de Retroalimentación Local (RL Interno) ---
class RlNeuron(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        # Pesos y sesgo estándar de la neurona
        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # Parámetro para el "RL Interno"
        # Este factor de adaptabilidad local se ajustará.
        # Lo inicializamos en 1.0 para que no tenga efecto al inicio.
        self.local_adapt_factor = nn.Parameter(torch.ones(output_features) * 0.1) # Un factor por neurona de salida

        # Variable para almacenar el gradiente recibido en la última corrección.
        # No es un Parameter, ya que se calcula en el forward o en el hook.
        self._last_weight_grad_norm = None

        # Registramos un hook para capturar el gradiente de los pesos después del backward pass
        # Este hook se ejecutará DESPUÉS de que .backward() calcule el gradiente para self.weight
        self.weight.register_hook(self._capture_grad_norm)

    def _capture_grad_norm(self, grad):
        # Esta función se llama cuando se calcula el gradiente para self.weight
        # Usamos .detach() para no interferir con el cálculo de gradientes normal
        self._last_weight_grad_norm = grad.norm().item() # Norma L2 del gradiente del peso

    def forward(self, x):
        # La salida de la neurona se multiplica por el factor de adaptabilidad local
        # Esto es una forma de simular el impacto del RL interno en la salida
        # Podrías hacer que afecte la función de activación, el sesgo, etc.
        linear_output = torch.matmul(x, self.weight) + self.bias

        # Aplicamos el factor de adaptabilidad local *después* de la operación lineal
        # Esto significa que la salida de la neurona está "influenciada" por su estado RL
        # F.relu se aplica para mantener la no linealidad
        output = F.relu(linear_output * self.local_adapt_factor)

        return output

    # Este método se llamaría DESPUÉS de optimizer.step()
    # Aquí es donde el "RL interno" entra en juego y corrige su propio estado
    def apply_local_feedback(self, lr_local_feedback=0.01):
        if self._last_weight_grad_norm is not None:
            # Una lógica simple de RL: si el gradiente fue grande, aumenta el factor de adaptabilidad.
            # Podrías usar funciones de recompensa, umbrales, etc.
            # Aquí, simplemente ajustamos el factor en proporción a la magnitud del gradiente del peso
            # y a una tasa de aprendizaje para el RL.
            # Usamos item() para obtener el valor numérico del tensor escalar.

            # Ajustamos el local_adapt_factor
            # NOTA: Directamente modificar self.local_adapt_factor.data
            # evita que esta operación sea parte del grafo de computación principal,
            # lo cual es deseable para un "meta-ajuste" como este.
            adjustment = self._last_weight_grad_norm * lr_local_feedback

            # Podemos limitar el rango del factor para evitar inestabilidad
            self.local_adapt_factor.data.add_(adjustment).clamp_(min=0.01, max=10.0)

            # Reiniciamos el gradiente capturado para la siguiente iteración
            self._last_weight_grad_norm = None


# --- Red Neuronal usando la Neurona con RL Interno ---
class RlNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rl_neuron1 = RlNeuron(input_size, hidden_size)
        self.rl_neuron2 = RlNeuron(hidden_size, hidden_size)
        # La capa de salida puede ser una neurona normal o también una RLNeuron
        # Para simplificar, usaremos una lineal estándar al final
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.rl_neuron1(x)
        h2 = self.rl_neuron2(h1)
        out = self.output_layer(h2)
        return out

    def apply_all_local_feedback(self, lr_local_feedback=0.01):
        # Itera sobre todas las neuronas RL y aplica su retroalimentación local
        self.rl_neuron1.apply_local_feedback(lr_local_feedback)
        self.rl_neuron2.apply_local_feedback(lr_local_feedback)

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h2)) # ERROR: Debería ser h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE_GLOBAL = 0.01
LEARNING_RATE_LOCAL_FEEDBACK = 0.005 # Tasa de aprendizaje para el RL interno
EPOCHS = 1000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 16 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización de Modielos ---
print("--- Inicializando y Entrenando Red con Neuronas RL Internas ---")
rl_model = RlNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_rl = torch.optim.Adam(rl_model.parameters(), lr=LEARNING_RATE_GLOBAL)
historial_loss_rl = []

total_params_rl = sum(p.numel() for p in rl_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red con Neuronas RL: {total_params_rl}")

for epoch in range(EPOCHS):
    rl_model.train()
    optimizer_rl.zero_grad()

    y_pred = rl_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_rl.append(loss.item())

    loss.backward() # Calcula los gradientes, los hooks se activan aquí

    # Aplicar el bucle de retroalimentación local DESPUÉS de que los gradientes han sido calculados
    # y ANTES de que el optimizador global actualice los pesos.
    # O, más comúnmente, DESPUÉS de optimizer.step() para que el ajuste local sea sobre los pesos ya corregidos.
    # Optaremos por DESPUÉS para que el RL interno reaccione a la corrección FINAL del optimizador.
    optimizer_rl.step()
    rl_model.apply_all_local_feedback(lr_local_feedback=LEARNING_RATE_LOCAL_FEEDBACK)

    if epoch % 1000 == 0:
        print(f"Red RL - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red RL): {historial_loss_rl[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar ---")
# Corrección en StandardNet: h2 = F.relu(self.fc2(h1))
class StandardNetCorrected(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1)) # Corregido: usa h1 como entrada
        out = self.output_layer(h2)
        return out


standard_model = StandardNetCorrected(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE_GLOBAL)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 1000 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

rl_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_rl = rl_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_rl, label="Predicción Red RL", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_rl, label="Pérdida Red RL", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Neuronas RL): {historial_loss_rl[-1]:.6f}")
print(f"Pérdida final (Red Estándar):         {historial_loss_standard[-1]:.6f}")

## Entorno RL para código Python

Medido en el banco (`laboratorio/banco/refuerzo_06.py`, `resultados-06-refuerzo.md`): «hola mundo» es imposible porque el espacio no está entre los caracteres; con «hola», PPO tampoco lo logra en 300 000 pasos. La demostración usaba `reset`/`step` del entorno vectorizado con la forma de Gymnasium y se caía.

In [ ]:
!pip install stable_baselines3

In [ ]:
import random
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# Importa los componentes necesarios de Stable Baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecEnvWrapper

# --- 1. Entorno Base (RawPythonEnv) ---
# Este entorno solo maneja la lógica del estado y las transiciones.
# NO calcula recompensas directamente.
class RawPythonEnv(gym.Env):
    metadata = {'render_modes': ['human'], 'render_fps': 4}

    def __init__(self, target_output="hola", max_code_tokens=15):
        super().__init__()
        self.target_output = target_output
        self.max_code_tokens = max_code_tokens
        self.current_tokens = []
        self.current_output = ""
        self.defined_vars = {}
        self.last_parsed_line = ""
        self.last_line_error = False  # Bandera para indicar si la última línea fue un error

        self.alphabet = 'abcdefghijklmnopqrstuvwxyz'
        self.digits = '0123456789'
        self.operators = ['+', '-', '=', '"', '(', ')', '_'] # Añadido '_' para nombres de variables
        self.keywords = ['print']
        self.action_space_chars = list(self.alphabet) + list(self.digits) + self.operators + self.keywords + ['_EOS_']
        self.action_space = spaces.Discrete(len(self.action_space_chars))

        # Espacio de observación:
        # One-hot de los últimos 5 tokens + presencia de variables + longitud de salida + coincidencia con objetivo
        self.observation_space = spaces.Box(low=0, high=1, shape=(5 * len(self.action_space_chars) + len(self.alphabet) + 2,), dtype=np.float32)

    def _get_obs(self):
        # One-hot encoding de los últimos 5 tokens para dar contexto sintáctico
        obs_tokens = np.zeros(5 * len(self.action_space_chars))
        for i, token in enumerate(self.current_tokens[-5:]):
            if token in self.action_space_chars:
                token_idx = self.action_space_chars.index(token)
                obs_tokens[i * len(self.action_space_chars) + token_idx] = 1

        # Presencia de variables definidas (a-z)
        vars_presence = np.array([1 if var in self.defined_vars else 0 for var in self.alphabet], dtype=np.float32)

        # Longitud de la salida actual y si contiene el objetivo
        output_len = np.array([len(self.current_output) / 100.0], dtype=np.float32)  # Normalizar
        target_match = np.array([1.0 if self.target_output in self.current_output else 0.0], dtype=np.float32)

        obs = np.concatenate([obs_tokens, vars_presence, output_len, target_match])
        return obs

    def _get_info(self):
        # Esta información se pasará al algoritmo de recompensa externo
        return {
            "code_tokens": self.current_tokens,
            "output": self.current_output,
            "defined_vars": self.defined_vars,
            "last_parsed_line": self.last_parsed_line,
            "last_line_error": self.last_line_error,
            "target_output": self.target_output,  # Se pasa el objetivo para el cálculo de recompensa
            "current_token_count": len(self.current_tokens) # Cuántos tokens se han generado en la línea actual
        }

    def _interpret_line(self, line_tokens):
        # Simulación de un intérprete Python básico
        line = "".join(line_tokens)
        temp_output = ""
        error = False

        if line.startswith("print("):
            if line.endswith(")"):
                content = line[6:-1]
                if content.startswith("\"") and content.endswith("\""):
                    try:
                        temp_output = content[1:-1]
                    except Exception:
                        error = True
                elif '+' in content:
                    parts = content.split('+')
                    if len(parts) == 2:
                        var1_name = parts[0].strip()
                        var2_name = parts[1].strip()
                        if var1_name in self.defined_vars and var2_name in self.defined_vars:
                            try:
                                res = self.defined_vars[var1_name] + self.defined_vars[var2_name]
                                temp_output = str(res)
                            except TypeError:
                                error = True
                        else:
                            error = True
                    else:
                        error = True
                elif '-' in content:
                    parts = content.split('-')
                    if len(parts) == 2:
                        var1_name = parts[0].strip()
                        var2_name = parts[1].strip()
                        if var1_name in self.defined_vars and var2_name in self.defined_vars:
                            if isinstance(self.defined_vars[var1_name], (int, float)) and isinstance(self.defined_vars[var2_name], (int, float)):
                                res = self.defined_vars[var1_name] - self.defined_vars[var2_name]
                                temp_output = str(res)
                            else:
                                error = True
                        else:
                            error = True
                    else:
                        error = True
                elif content in self.defined_vars:
                    temp_output = str(self.defined_vars[content])
                else:
                    error = True
            else:
                error = True
        elif '=' in line:  # Asignación de variables
            parts = line.split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                value_str = parts[1].strip()
                if var_name in self.alphabet:
                    if value_str.startswith("\"") and value_str.endswith("\""):
                        self.defined_vars[var_name] = value_str[1:-1]
                    elif value_str.isdigit():
                        self.defined_vars[var_name] = int(value_str)
                    elif '+' in value_str:
                        sub_parts = value_str.split('+')
                        if len(sub_parts) == 2:
                            val1_str = sub_parts[0].strip()
                            val2_str = sub_parts[1].strip()
                            val1 = self.defined_vars.get(val1_str) if val1_str in self.defined_vars else (int(val1_str) if val1_str.isdigit() else None)
                            val2 = self.defined_vars.get(val2_str) if val2_str in self.defined_vars else (int(val2_str) if val2_str.isdigit() else None)

                            if val1 is not None and val2 is not None:
                                try: self.defined_vars[var_name] = val1 + val2
                                except TypeError: error = True
                            else: error = True
                        else: error = True
                    elif '-' in value_str:
                        sub_parts = value_str.split('-')
                        if len(sub_parts) == 2:
                            val1_str = sub_parts[0].strip()
                            val2_str = sub_parts[1].strip()
                            val1 = self.defined_vars.get(val1_str) if val1_str in self.defined_vars else (int(val1_str) if val1_str.isdigit() else None)
                            val2 = self.defined_vars.get(val2_str) if val2_str in self.defined_vars else (int(val2_str) if val2_str.isdigit() else None)
                            if val1 is not None and val2 is not None:
                                if isinstance(val1, (int, float)) and isinstance(val2, (int, float)):
                                    self.defined_vars[var_name] = val1 - val2
                                else: error = True
                            else: error = True
                        else: error = True
                    elif value_str in self.defined_vars:
                        self.defined_vars[var_name] = self.defined_vars[value_str]
                    else: error = True
                else: error = True
            else: error = True
        else:
            error = True  # Sintaxis no reconocida

        return temp_output, error

    def step(self, action):
        terminated = False  # Si el episodio termina por éxito o fallo
        truncated = False   # Si el episodio termina por límite de tiempo/pasos

        token_to_add = self.action_space_chars[action]
        self.current_tokens.append(token_to_add)
        self.last_line_error = False  # Resetear la bandera de error para la línea actual

        # Si el agente genera un token de fin de línea, intentar interpretar la línea
        if token_to_add == '_EOS_':
            self.last_parsed_line = "".join(self.current_tokens[:-1])  # La línea sin el _EOS_
            output, error = self._interpret_line(self.current_tokens[:-1])  # Interpretar hasta el _EOS_

            if not error:
                self.current_output += output
            else:
                self.last_line_error = True  # Marcar que hubo un error

            self.current_tokens = []  # Reiniciar tokens para la siguiente línea

        # Condiciones de terminación del episodio (independientes de la recompensa)
        if self.target_output in self.current_output:
            terminated = True # El objetivo se logró
        elif len(self.current_tokens) + len(self.last_parsed_line) >= self.max_code_tokens * 2 and '_EOS_' not in self.current_tokens:
            # Si el agente escribe demasiados tokens en una sola línea sin un _EOS_
            truncated = True
        elif len(self.current_output) > len(self.target_output) * 2 and not (self.target_output in self.current_output):
            # Si la salida acumulada crece demasiado sin alcanzar el objetivo
            truncated = True
        elif len(self.current_tokens) >= self.max_code_tokens * 2 and not terminated:
            # Si el número total de tokens en la línea actual excede el límite
            truncated = True

        obs = self._get_obs()
        info = self._get_info()

        # IMPORTANTE: La recompensa aquí es SIEMPRE 0.0. Será calculada por el algoritmo externo.
        return obs, 0.0, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed, options=options)  # Asegura compatibilidad con Gymnasium
        self.current_tokens = []
        self.current_output = ""
        self.defined_vars = {}
        self.last_parsed_line = ""
        self.last_line_error = False
        obs = self._get_obs()
        info = self._get_info()
        return obs, info

    def render(self, mode='human'):
        if mode == 'human':
            print("\n--- AGENTE CONSTRUYENDO LÍNEA (Entorno Base) ---")
            print("".join(self.current_tokens))
            if self.last_parsed_line:
                print(f"ÚLTIMA LÍNEA INTERPRETADA: {self.last_parsed_line}")
            print(f"SALIDA ACUMULADA: '{self.current_output}'")
            print(f"VARIABLES: {self.defined_vars}")
            print(f"ERROR EN ÚLTIMA LÍNEA: {self.last_line_error}")
            print("---------------------------------")


# --- 2. Algoritmo Externo de Recompensa (calculate_custom_reward) ---
# Esta función es el "otro algoritmo" que decide la recompensa y el castigo.
def calculate_custom_reward(info_from_env: dict, action_taken: int, eos_idx: int, max_code_tokens: int):
    reward = -0.005  # Pequeña penalización por cada token/paso (fomenta la eficiencia)

    # Solo aplicamos recompensas/castigos relacionados con la línea cuando se ha interpretado una línea completa
    # Se detecta que se procesó una línea si current_tokens está vacío y last_parsed_line no lo está
    if not info_from_env["code_tokens"] and info_from_env["last_parsed_line"]:
        if info_from_env["last_line_error"]:
            reward -= 1.0  # **Castigo fuerte** por error de sintaxis en la línea
        else:
            reward += 0.5  # **Recompensa** por una línea sintácticamente válida

            # Recompensas/castigos basados en el objetivo de salida
            if info_from_env["output"] == info_from_env["target_output"]:
                reward += 5.0  # **Gran recompensa** por lograr el objetivo exacto
            elif info_from_env["target_output"] in info_from_env["output"]:
                reward += 2.0  # **Recompensa** por contener el objetivo (progreso hacia la meta)

    # Penalización adicional si el agente excede el número máximo de tokens por línea sin un _EOS_
    if action_taken != eos_idx and info_from_env["current_token_count"] >= max_code_tokens:
        reward -= 0.5 # Pequeña penalización por "divagar" demasiado sin cerrar la línea

    return reward

# --- 3. Wrapper que Inyecta la Recompensa (RewardInjectingWrapper) ---
# Este wrapper se interpone entre el agente y el entorno base para modificar la recompensa.
class RewardInjectingWrapper(gym.Wrapper):
    def __init__(self, env, reward_calculator_func):
        super().__init__(env)
        self.reward_calculator = reward_calculator_func
        # Necesitamos pasar el índice de _EOS_ y max_code_tokens al calculador de recompensa si la usa
        self._eos_idx = self.env.action_space_chars.index('_EOS_')
        self._max_code_tokens = self.env.max_code_tokens

    def step(self, action):
        # Llama al método `step()` del entorno base (RawPythonEnv), que devuelve reward=0.0
        observation, base_reward, terminated, truncated, info = self.env.step(action)

        # Calcula la recompensa real utilizando el algoritmo externo
        # Pasa la información del entorno, la acción tomada, el índice _EOS_ y el max_code_tokens
        custom_reward = self.reward_calculator(info, action, self._eos_idx, self._max_code_tokens)

        # Devuelve la observación, la recompensa CALCULADA por el algoritmo externo, y el resto de la info
        return observation, custom_reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        return self.env.reset(seed=seed, options=options)


# --- 4. Script Principal de Entrenamiento y Demostración ---
if __name__ == "__main__":
    TARGET_OUTPUT = "hola mundo"
    MAX_CODE_TOKENS_PER_LINE = 20 # Límite de tokens para una sola línea de código

    # Crear el entorno base (que NO calcula recompensa)
    base_env_instance = RawPythonEnv(target_output=TARGET_OUTPUT, max_code_tokens=MAX_CODE_TOKENS_PER_LINE)

    # Definir la función que "make_vec_env" usará para crear instancias de entornos
    # Ahora esta function creará un entorno envuelto en Monitor Y luego en RewardInjectingWrapper
    def make_wrapped_env():
        # Envolver el entorno base con nuestro RewardInjectingWrapper
        wrapped_with_reward = RewardInjectingWrapper(base_env_instance, calculate_custom_reward)
        # Opcional: Envolver con Monitor para ver métricas de entrenamiento en TensorBoard
        # Apply Monitor *before* make_vec_env vectorizes it
        monitored_env = Monitor(wrapped_with_reward)
        return monitored_env # Return the environment wrapped in both

    # Crear un entorno vectorizado (aunque solo sea uno para simplificar)
    # make_vec_env se encargará de llamar a make_wrapped_env n_envs veces
    env = make_vec_env(make_wrapped_env, n_envs=1)

    # El Monitor ya está aplicado dentro de make_wrapped_env,
    # así que no necesitamos aplicarlo de nuevo aquí:
    # env = Monitor(env) # REMOVER esta línea

    # Configuración del Agente PPO (modelo fin a fin)
    # PPO aprende directamente de las observaciones a las acciones, recibiendo recompensas
    # que fueron calculadas por nuestro algoritmo externo.
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, n_steps=2048,
                batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2,
                ent_coef=0.01, # Fomenta la exploración
                policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])), # Redes neuronales más grandes
                tensorboard_log="./ppo_external_reward_tensorboard/")

    print(f"\n--- Entrenando el Agente PPO para generar: '{TARGET_OUTPUT}' ---")
    model.learn(total_timesteps=300000) # Aumenta los pasos para una tarea más compleja
    print("Entrenamiento completado.")

    # --- Demostración del Agente Aprendido ---
    print("\n--- Demostrando el Agente Aprendido (con recompensa externa) ---")

    # Resetear el entorno para la demostración
    # Aquí ya no necesitamos el manejo especial de errores de 'reset()'
    # porque `make_vec_env` y los wrappers lo hacen robusto.
    obs, info = env.reset()

    done = False
    episode_reward = 0
    steps = 0
    print("\n--- AGENTE CONSTRUYENDO Y EJECUTANDO ---")

    # Acceder a la instancia del entorno RawPythonEnv dentro del wrapper
    # Esto es para obtener acceso a atributos como action_space_chars directamente
    # Dado que tenemos Monitor y RewardInjectingWrapper, debemos "desenvolver" dos veces
    # The env object is a DummyVecEnv, which contains the Monitor wrapper,
    # which contains the RewardInjectingWrapper, which contains the RawPythonEnv.
    # So, we need to access env.envs[0].env.env to get the original env instance
    unwrapped_raw_env = env.envs[0].env.env # Assumes n_envs=1


    while not done and steps < unwrapped_raw_env.max_code_tokens * 5: # Límite de pasos para la demo
        action, _states = model.predict(obs, deterministic=True) # Elegir la acción más segura
        # El método step de 'env' (que es el DummyVecEnv con wrappers dentro)
        # devolverá la recompensa calculada por 'calculate_custom_reward'
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated[0] or truncated[0] # terminated and truncated are lists for VecEnvs
        episode_reward += reward[0] # reward is also a list for VecEnvs
        steps += 1

        # Acceder a la información del primer (y único) entorno en la lista de infos
        current_info = info[0]

        # Imprimir el token que el agente añadió
        print(f"Token añadido: '{unwrapped_raw_env.action_space_chars[action.item()]}'") # .item() para desempacar array de 1 elemento

        # Si la línea ha sido parseada (cuando el agente añade '_EOS_')
        # Check if current_info["code_tokens"] is empty and last_parsed_line exists
        if not current_info["code_tokens"] and current_info.get("last_parsed_line"):
            print(f"  LÍNEA INTERPRETADA: '{current_info['last_parsed_line']}'")
            print(f"  SALIDA ACUMULADA: '{current_info['output']}'")
            print(f"  VARIABLES: {current_info['defined_vars']}")
            # The reward printed here is the cumulative reward within the Monitor wrapper
            # For the step-by-step reward, you would access info[0]['episode']['r'] or similar if Monitor records step rewards
            # For simplicity here, we'll print the cumulative reward or the step reward from the step() output
            print(f"  RECOMPENSA DE ESTE PASO: {reward[0]:.4f}") # Print the step reward from the step output
            print("---------------------------------")
            # Resetear la línea interpretada en la info para no repetirla en la impresión
            # Modifying the info dictionary like this won't affect the actual info passed to the next step by the VecEnv/Monitor
            # This is purely for the print statement's logic.
            current_info["last_parsed_line"] = ""

    print(f"\n--- DEMO FINALIZADA ---")
    # Access the final output from the info dictionary after the episode ends
    # This might be slightly different depending on how Monitor aggregates info
    # A safer way is to track the output during the loop or access it directly from the unwrapped env if needed
    final_output_info = info[0] if done else env.envs[0].env.get_info()[0] # Try to get info after episode or last one
    print(f"SALIDA FINAL: '{final_output_info.get('output', 'N/A')}'")
    print(f"RECOMPENSA TOTAL DEL EPISODIO: {episode_reward:.2f}") # episode_reward tracked during the loop

    env.close()

## Red de votación de decisiones (RL)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.embed = nn.Linear(input_dim, embed_dim)

    def forward(self, x):
        return torch.tanh(self.embed(x))

class DecisionHead(nn.Module):
    def __init__(self, embed_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 64)
        self.fc2 = nn.Linear(64, action_dim) # action_dim representa aquí el número de opciones/clases

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class DecisionVotingNet(nn.Module):
    def __init__(self, input_dim, embed_dim, action_dim, num_heads=4):
        super().__init__()
        self.embedding = EmbeddingNetwork(input_dim, embed_dim)
        self.heads = nn.ModuleList([DecisionHead(embed_dim, action_dim) for _ in range(num_heads)])

    def forward(self, x):
        z = self.embedding(x)
        logits_list = [head(z) for head in self.heads]
        return logits_list

# Definir las posibles acciones/frases (esto es crucial para tu ejemplo)
# Deben corresponder con action_dim
POSSIBLE_PHRASES = [
    "durmiendo",
    "sonando",
    "comiendo",
    "jugando",
    "observando"
]

if __name__ == "__main__":
    action_dim = len(POSSIBLE_PHRASES) # El número de posibles acciones/frases
    model = DecisionVotingNet(input_dim=10, embed_dim=32, action_dim=action_dim, num_heads=4)
    dummy_input = torch.randn(1, 10) # Un solo ejemplo de entrada

    individual_logits = model(dummy_input)

    print("Generación (predicción) de cada cabeza:")
    for i, logits in enumerate(individual_logits):
        probabilities = F.softmax(logits, dim=-1) # Convertir logits a probabilidades

        # Encontrar la acción/frase con la probabilidad más alta
        predicted_index = torch.argmax(probabilities, dim=-1).item()
        predicted_phrase = POSSIBLE_PHRASES[predicted_index]
        confidence = probabilities[0, predicted_index].item() # Probabilidad de la frase elegida

        print(f"Cabeza {i+1}: El perro sigue {predicted_phrase} (Confianza: {confidence:.2f})")

    print(f"\nNúmero de cabezas (salidas): {len(individual_logits)}")

#Auto-Optimización por Sacarina

1. Estado s
2. Acción a
3. Recompensa r
4. Transición T(s, a)
5. Meta final

6. Criterios de éxito y exploración
5. RL Autoimplosivo ("RL Entropía Negativa")

🧬 Concepto:

El agente comienza con una red neuronal grande. Cada acción le cuesta un trozo de su red. Debe resolver tareas mientras se auto-descompone.

🧪 Mecánica:

Cada decisión requiere apagar nodos o capas enteras.

Recompensa si logra seguir resolviendo tareas con menor estructura.

Penalización si se vuelve inútil demasiado pronto.


🔍 Objetivo:

Estudiar eficiencia mínima, plasticidad y adaptación extrema.

🧩 Único porque:

Es un RL que reduce sus propios grados de libertad como precio por actuar. Al final solo queda un agente degenerado… o optimizado.

## Entorno visual multicuerpo

In [ ]:

!apt-get install -y xvfb x11-utils
!pip install pygame pyvirtualdisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(640, 480))
display.start()

In [ ]:
import pygame
import os

class VisualMultiBodyEnv(MultiBodyCubePushEnv):
    def __init__(self, render_interval=24, save_dir="frames"):
        super().__init__()
        self.render_interval = render_interval
        self.save_dir = save_dir
        self.frame_count = 0

        # Inicializar pygame en modo headless
        os.makedirs(self.save_dir, exist_ok=True)
        pygame.init()
        self.screen_size = 480
        self.scale = 100
        self.screen = pygame.Surface((self.screen_size, self.screen_size))

    def draw_circle(self, pos, color, radius=5):
        p = ((pos + 1) * 0.5 * self.screen_size).astype(int)
        pygame.draw.circle(self.screen, color, p, radius)

    def render(self):
        self.screen.fill((255, 255, 255))  # Fondo blanco

        # Dibujar objetos
        self.draw_circle(self.body1_pos, (255, 0, 0), radius=8)     # Rojo
        self.draw_circle(self.body2_pos, (0, 255, 0), radius=8)     # Verde
        self.draw_circle(self.cube_pos, (0, 0, 255), radius=10)     # Azul
        self.draw_circle(self.goal_pos, (255, 165, 0), radius=6)    # Naranja

        # Guardar imagen cada 24 pasos
        if self.frame_count % self.render_interval == 0:
            path = os.path.join(self.save_dir, f"frame_{self.frame_count:04d}.png")
            pygame.image.save(self.screen, path)
            print(f"🖼️ Saved {path}")

        self.frame_count += 1

In [ ]:
env = VisualMultiBodyEnv()
obs, _ = env.reset()

done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()

    if terminated or truncated:
        break

In [ ]:
import imageio.v2 as imageio
import os

images = []
folder = "frames"
for i in sorted(os.listdir(folder)):
    if i.endswith(".png"):
        images.append(imageio.imread(os.path.join(folder, i)))

imageio.mimsave("cube_push.gif", images, fps=10)

In [ ]:
from IPython.display import Image
Image(filename="cube_push.gif")

## Empujar cubos con varios cuerpos (RL)

Medido en el banco: tal cual no llega nunca (0 %) y los episodios no terminaban (faltaba un límite de pasos). Con una recompensa pequeña por acercarse al cubo, llega en el 65 % de los episodios.

In [ ]:
# --- 0. Instalación y Configuración Inicial (Necesario en Colab) ---
# Si aún no lo has ejecutado, asegúrate de hacerlo:
# !apt-get install -y xvfb x11-utils > /dev/null 2>&1
!pip install pygame pyvirtualdisplay > /dev/null 2>&1
# !pip install imageio[ffmpeg] > /dev/null 2>&1 # Necesario para guardar GIF/MP4
!pip install stable-baselines3 torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # Instala PyTorch con CUDA y SB3

from pyvirtualdisplay import Display
import os
import torch # Importar torch para verificar la disponibilidad de CUDA

# Iniciar un display virtual para pygame (DESCOMENTAR ESTO PARA EJECUTAR EN COLAB)
# print("Iniciando display virtual...")
# display = Display(visible=0, size=(640, 480))
# display.start()
# print("Display virtual iniciado.")

import pygame
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
import time

# --- 1. Tu Entorno MultiBodyCubePushEnv (con correcciones y mejoras) ---
class MultiBodyCubePushEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, render_mode=None):
        super().__init__()
        self.grid_size = 10
        self.render_mode = render_mode

        self.observation_space = spaces.Box(low=0, high=self.grid_size - 1, shape=(8,), dtype=np.int32)
        self.action_space = spaces.MultiDiscrete([5, 5])

        self.body1_pos = None
        self.body2_pos = None
        self.cube_pos = None
        self.goal_pos = np.array([self.grid_size - 1, self.grid_size - 1])

        # Atributos para Pygame render
        self.screen = None
        self.clock = None
        self.cell_size = 50 # Tamaño en píxeles de cada celda

        # Atributo para la recompensa de progreso
        self.last_distance_to_goal = None

    def _get_obs(self):
        return np.concatenate([
            self.body1_pos, self.body2_pos, self.cube_pos, self.goal_pos
        ]).astype(np.int32)

    def _get_info(self):
        return {"distance_to_goal": np.linalg.norm(self.cube_pos - self.goal_pos)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        while np.array_equal(self.body1_pos, self.body2_pos) or \
              np.array_equal(self.body1_pos, self.cube_pos) or \
              np.array_equal(self.body2_pos, self.cube_pos) or \
              np.array_equal(self.cube_pos, self.goal_pos):
            self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        observation = self._get_obs()
        self.last_distance_to_goal = np.linalg.norm(self.cube_pos - self.goal_pos) # Inicializar para recompensa de progreso
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info

    def step(self, action):
        action_body1, action_body2 = action

        def apply_move(pos, a):
            new_pos = pos.copy()
            if a == 0: new_pos[1] -= 1  # Arriba
            elif a == 1: new_pos[1] += 1  # Abajo
            elif a == 2: new_pos[0] -= 1  # Izquierda
            elif a == 3: new_pos[0] += 1  # Derecha
            # Si a == 4, no se mueve

            new_pos = np.clip(new_pos, 0, self.grid_size - 1)
            return new_pos

        prev_body1_pos = self.body1_pos.copy()
        prev_body2_pos = self.body2_pos.copy()

        # Mover cuerpos a sus nuevas posiciones propuestas
        proposed_body1_pos = apply_move(self.body1_pos, action_body1)
        proposed_body2_pos = apply_move(self.body2_pos, action_body2)

        # Determinar si cada cuerpo quiere moverse a la celda del cubo
        b1_wants_to_move_to_cube = np.array_equal(proposed_body1_pos, self.cube_pos)
        b2_wants_to_move_to_cube = np.array_equal(proposed_body2_pos, self.cube_pos)

        # Lógica de empuje del cubo
        new_cube_pos = self.cube_pos.copy()
        cube_moved = False

        # Delta de acciones a movimiento (para entender la dirección intentada)
        action_to_delta = {
            0: np.array([0,-1]), # Arriba
            1: np.array([0,1]),  # Abajo
            2: np.array([-1,0]), # Izquierda
            3: np.array([1,0]),  # Derecha
            4: np.array([0,0])   # No mover
        }
        delta_action_b1 = action_to_delta[action_body1]
        delta_action_b2 = action_to_delta[action_body2]

        # Condición de empuje: Ambos cuerpos quieren moverse a la posición del cubo Y sus acciones son en la misma dirección
        # (Esto simula que empujan juntos un objeto)
        if b1_wants_to_move_to_cube and b2_wants_to_move_to_cube:
            # Empuje en X
            if delta_action_b1[0] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_x = self.cube_pos[0] + delta_action_b1[0]
                if 0 <= potential_cube_move_x < self.grid_size: # No salirse de límites
                    new_cube_pos[0] = potential_cube_move_x
                    cube_moved = True
            # Empuje en Y
            if delta_action_b1[1] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_y = self.cube_pos[1] + delta_action_b1[1]
                if 0 <= potential_cube_move_y < self.grid_size: # No salirse de límites
                    new_cube_pos[1] = potential_cube_move_y
                    cube_moved = True

        if cube_moved:
            # Si el cubo se movió, los cuerpos que lo empujaron se mueven a la posición que ocupaba el cubo
            self.body1_pos = self.cube_pos # La antigua posición del cubo
            self.body2_pos = self.cube_pos # Asumiendo que ambos empujan y terminan en la misma celda
            self.cube_pos = new_cube_pos
        else:
            # Si el cubo no se movió, los cuerpos se mueven a sus posiciones propuestas,
            # pero no pueden ocupar la celda del cubo si ya está ocupada
            if not b1_wants_to_move_to_cube:
                self.body1_pos = proposed_body1_pos
            else: # Si b1 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body1_pos = prev_body1_pos

            if not b2_wants_to_move_to_cube:
                self.body2_pos = proposed_body2_pos
            else: # Si b2 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body2_pos = prev_body2_pos


        # Clip para asegurar que todo esté dentro de los límites
        self.body1_pos = np.clip(self.body1_pos, 0, self.grid_size - 1)
        self.body2_pos = np.clip(self.body2_pos, 0, self.grid_size - 1)
        self.cube_pos = np.clip(self.cube_pos, 0, self.grid_size - 1)

        reward = 0.0
        terminated = False
        truncated = False

        # Recompensa de progreso
        current_distance = np.linalg.norm(self.cube_pos - self.goal_pos)
        reward = -0.1 # Pequeña penalización por cada paso
        reward += (self.last_distance_to_goal - current_distance) * 0.5 # Recompensa por acercarse a la meta
        self.last_distance_to_goal = current_distance # Actualizar para el siguiente paso

        # Recompensa por alcanzar la meta
        if np.array_equal(self.cube_pos, self.goal_pos):
            reward = 10.0 # Recompensa grande por alcanzar la meta
            terminated = True # El episodio termina

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _render_frame(self):
        # Asegurarse de que Pygame esté inicializado y la pantalla creada
        if self.screen is None:
            pygame.init()
            self.screen_size_px = self.grid_size * self.cell_size
            self.screen = pygame.display.set_mode((self.screen_size_px, self.screen_size_px))
            pygame.display.set_caption("Multi-Body Cube Push Env")
            self.clock = pygame.time.Clock()

        # Colores
        WHITE = (255, 255, 255)
        BLACK = (0, 0, 0)
        RED = (255, 0, 0)     # Cuerpo 1
        GREEN = (0, 255, 0)   # Cuerpo 2
        BLUE = (0, 0, 255)    # Cubo
        ORANGE = (255, 165, 0) # Meta

        self.screen.fill(WHITE) # Fondo blanco

        # Dibujar la cuadrícula
        for x in range(self.grid_size):
            for y in range(self.grid_size):
                rect = pygame.Rect(x * self.cell_size, y * self.cell_size, self.cell_size, self.cell_size)
                pygame.draw.rect(self.screen, BLACK, rect, 1) # Borde de celda

        # Función para dibujar objetos en el centro de la celda
        def draw_object(pos_grid, color, shape="circle", radius_ratio=0.8):
            center_x = (pos_grid[0] * self.cell_size) + (self.cell_size / 2)
            center_y = (pos_grid[1] * self.cell_size) + (self.cell_size / 2)
            radius = int(self.cell_size / 2 * radius_ratio)
            if shape == "circle":
                pygame.draw.circle(self.screen, color, (center_x, center_y), radius)
            elif shape == "rect":
                size = int(self.cell_size * radius_ratio)
                rect_pos_x = center_x - size / 2
                rect_pos_y = center_y - size / 2
                pygame.draw.rect(self.screen, color, (rect_pos_x, rect_pos_y, size, size))

        # Dibujar objetos
        draw_object(self.goal_pos, ORANGE, shape="circle", radius_ratio=0.6) # Meta (más pequeña)
        draw_object(self.cube_pos, BLUE, shape="rect", radius_ratio=0.8) # Cubo (rectángulo)
        draw_object(self.body1_pos, RED, shape="circle", radius_ratio=0.8) # Cuerpo 1
        draw_object(self.body2_pos, GREEN, shape="circle", radius_ratio=0.8) # Cuerpo 2

        pygame.event.pump()
        pygame.display.flip()

        if self.render_mode == "human" and self.clock:
            self.clock.tick(self.metadata["render_fps"])

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None
            self.clock = None

# --- Ahora, integraremos esto en el flujo de entrenamiento de Stable-Baselines3 ---

# Importaciones necesarias para Stable-Baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

print("Entorno 'MultiBodyCubePushEnv' con renderizado Pygame (corregido).")

# --- Callback para guardar frames para el GIF ---
class PygameRenderCallback(BaseCallback):
    def __init__(self, verbose=0, render_freq=1, save_dir="frames"):
        super(PygameRenderCallback, self).__init__(verbose)
        self.render_freq = render_freq
        self.save_dir = save_dir
        self.frame_count = 0
        os.makedirs(self.save_dir, exist_ok=True)
        self.images = []

    def _on_training_start(self) -> None:
        # Asegurarse de que el entorno exista y tenga render_mode "human"
        if hasattr(self.model.env, 'envs') and self.model.env.envs[0].render_mode == "human":
            # Acceder al entorno base a través del wrapper Monitor (.env)
            self.model.env.envs[0].env._render_frame()

    def _on_step(self) -> bool:
        # Verificar si estamos en el entorno base (no vectorizado si n_envs=1)
        # y si el render_mode es "human" y si es el momento de renderizar
        if self.n_calls % self.render_freq == 0 and \
           hasattr(self.model.env, 'envs') and \
           self.model.env.envs[0].render_mode == "human":

            # Acceder al entorno base para la pantalla de Pygame
            env_instance = self.model.env.envs[0].env
            env_instance._render_frame() # Asegurarse de que el frame se dibuje

            # Capturar la pantalla de Pygame para el GIF
            img_array = pygame.surfarray.array3d(env_instance.screen)
            img_array = np.transpose(img_array, (1, 0, 2)) # Pygame es (W, H, C), necesitamos (H, W, C)
            self.images.append(img_array)

        self.frame_count += 1
        return True

    def _on_training_end(self) -> None:
        if hasattr(self.model.env, 'envs') and self.model.env.envs[0].render_mode == "human":
            self.model.env.envs[0].env.close() # Acceder al entorno base para cerrar
            if self.images:
                gif_path = os.path.join(self.save_dir, "training_animation.gif")
                print(f"\nGenerando GIF de entrenamiento en: {gif_path}")
                # Ajustar fps si es muy rápido
                imageio.mimsave(gif_path, self.images, fps=self.model.env.envs[0].env.metadata["render_fps"])
                print("GIF de entrenamiento generado.")


# --- Definición del dispositivo (GPU o CPU) ---
if torch.cuda.is_available():
    device = "cuda"
    print("GPU (CUDA) disponible. El entrenamiento se ejecutará en la GPU.")
else:
    device = "cpu"
    print("GPU (CUDA) NO disponible. El entrenamiento se ejecutará en la CPU.")


# --- Creación del entorno vectorizado para Stable-Baselines3 ---
# make_vec_env envuelve el entorno con Monitor por defecto.
env = make_vec_env(MultiBodyCubePushEnv, n_envs=1, env_kwargs={"render_mode": "human"}, seed=0)

# --- Configuración y Entrenamiento del Modelo PPO ---
print("\nIniciando entrenamiento del agente PPO con renderizado Pygame...")
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.001, n_steps=1024, batch_size=64,
            gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
            tensorboard_log="./discrete_ppo_tensorboard_logs/",
            device=device) # <-- ¡Aquí se especifica el dispositivo!

total_timesteps = 100_000
print(f"Entrenando por {total_timesteps} timesteps...")
# Pasamos el callback de PygameRender para capturar frames durante el entrenamiento
model.learn(total_timesteps=total_timesteps, callback=PygameRenderCallback(render_freq=1))

print("\nEntrenamiento finalizado.")
model.save("discrete_multi_body_cube_push_ppo_pygame")
print("Modelo guardado como 'discrete_multi_body_cube_push_ppo_pygame.zip'")

# --- Prueba del Modelo Entrenado (Demostración y GIF de Demostración) ---
print("\nDemostración del modelo entrenado:")
demo_images = []
obs, info = env.reset() # Resetear el entorno vectorizado

episodes = 3
for episode in range(episodes):
    done = False
    truncated = False
    episode_reward = 0
    while not done and not truncated:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        # Para la demostración, directamente llamar al render del entorno base y capturar
        env_instance = env.envs[0].env # Acceder al entorno base para la demostración
        env_instance._render_frame()
        img_array = pygame.surfarray.array3d(env_instance.screen)
        img_array = np.transpose(img_array, (1, 0, 2)) # Pygame es (W, H, C), necesitamos (H, W, C)
        demo_images.append(img_array)

        if terminated or truncated:
            print(f"Episodio Demostración {episode + 1}: Recompensa = {episode_reward:.2f}, Distancia final a meta: {info[0]['distance_to_goal']:.2f}")
            obs, info = env.reset() # Resetear el entorno vectorizado
            break

env.close() # Cierra el entorno de renderizado al final

# Generar GIF de la demostración
if demo_images:
    demo_gif_path = "cube_push_demo.gif"
    print(f"\nGenerando GIF de demostración en: {demo_gif_path}")
    imageio.mimsave(demo_gif_path, demo_images, fps=env.envs[0].env.metadata["render_fps"])
    print("GIF de demostración generado.")

    # Mostrar el GIF en Colab (descomentar para ejecutar en Colab)
    # from IPython.display import Image
    # try:
    #     display_instance = Display(visible=0, size=(640, 480))
    #     display_instance.start()
    #     display(Image(filename=demo_gif_path))
    #     display_instance.stop()
    # except Exception as e:
    #     print(f"No se pudo mostrar el GIF directamente: {e}")
    #     print(f"Puedes descargar el GIF desde el panel de archivos de Colab: {demo_gif_path}")

#Ann Biomas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición del Modelo NeuralBiomesModel (como proporcionado) ---
class NeuralBiomesModel(nn.Module):
    def __init__(self, input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim):
        super().__init__()

        # 1. Capa de Entrada Común ("El Mundo")
        self.shared_feature_extractor = nn.Sequential(
            nn.Linear(input_dim, shared_features_dim),
            nn.ReLU(),
            nn.Linear(shared_features_dim, shared_features_dim)
        )

        # 2. Bioma de Almacenamiento (LSTM para "memoria" secuencial)
        self.memory_biom = nn.LSTM(shared_features_dim, memory_hidden_dim, batch_first=True)

        # 3. Bioma de Procesamiento (MLP para lógica/cálculo)
        self.processing_biom = nn.Sequential(
            nn.Linear(shared_features_dim + memory_hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, processing_output_dim),
            nn.ReLU()
        )

        # 4. Capa de Salida Final (combinando resultados de los biomas)
        self.output_layer = nn.Linear(processing_output_dim, output_dim)

    def forward(self, x):
        # Paso 1: Extraer características en el "Mundo"
        shared_features = self.shared_feature_extractor(x)

        # Paso 2: Pasar características al Bioma de Almacenamiento
        shared_features_seq = shared_features.unsqueeze(1)
        memory_output, (h_n, c_n) = self.memory_biom(shared_features_seq)
        memory_state = h_n.squeeze(0)

        # Paso 3: Pasar características y memoria al Bioma de Procesamiento
        combined_features = torch.cat((shared_features, memory_state), dim=1)
        processed_output = self.processing_biom(combined_features)

        # Paso 4: Pasar el resultado procesado a la capa de salida final
        final_output = self.output_layer(processed_output)

        return final_output

# --- 2. Generación de Datos con make_circles ---
# Generar 1000 muestras con ruido y un factor de escala para el círculo interno
X, y = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)

# Convertir a tensores de PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Asegurar dimensión correcta para pérdida

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# --- 3. Inicialización del Modelo, Función de Pérdida y Optimizador ---
input_dim = X.shape[1] # 2 características (x, y)
shared_features_dim = 64
memory_hidden_dim = 32
processing_output_dim = 64
output_dim = 1 # Para clasificación binaria (0 o 1)

model = NeuralBiomesModel(input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim)
criterion = nn.BCEWithLogitsLoss() # Para clasificación binaria, maneja sigmoid internamente
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- 4. Entrenamiento del Modelo ---
num_epochs = 200
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(num_epochs):
    model.train() # Poner el modelo en modo entrenamiento
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval() # Poner el modelo en modo evaluación
    with torch.no_grad():
        # Evaluación en entrenamiento
        train_outputs = model(X_train)
        train_preds = (torch.sigmoid(train_outputs) > 0.5).float()
        train_acc = accuracy_score(y_train.numpy(), train_preds.numpy())
        train_accuracies.append(train_acc)

        # Evaluación en prueba
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)
        test_losses.append(test_loss.item())
        test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
        test_acc = accuracy_score(y_test.numpy(), test_preds.numpy())
        test_accuracies.append(test_acc)

    if (epoch + 1) % 20 == 0:
        print(f'Época [{epoch+1}/{num_epochs}], Pérdida Entrenamiento: {loss.item():.4f}, Precisión Entrenamiento: {train_acc:.4f}, Pérdida Prueba: {test_loss.item():.4f}, Precisión Prueba: {test_acc:.4f}')

# --- 5. Visualización de Datos make_circles ---
print("\n--- Visualización de Datos make_circles ---")
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label='Clase 0', alpha=0.7, edgecolors='w')
plt.scatter(X[y == 1, 0], X[y == 1, 1], label='Clase 1', alpha=0.7, edgecolors='w')
plt.title('Datos generados con make_circles')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.legend()
plt.grid(True)
plt.show()

# --- 6. Visualización de Gráficas de Entrenamiento y Prueba ---
print("\n--- Gráficas de Entrenamiento y Prueba ---")
epochs_list = range(1, num_epochs + 1)

# Gráfica de Pérdida
plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_losses, 'b', label='Pérdida de Entrenamiento')
plt.plot(epochs_list, test_losses, 'r', label='Pérdida de Prueba')
plt.title('Pérdida de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

# Gráfica de Precisión
plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_accuracies, 'b', label='Precisión de Entrenamiento')
plt.plot(epochs_list, test_accuracies, 'r', label='Precisión de Prueba')
plt.title('Precisión de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

##RL cerebro

##🧠 Prototipo de RL + Cerebro Modular con MoE

In [ ]:
# 🧠 Prototipo de RL + Cerebro Modular con MoE

import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# ---------- Configuracion base ----------
EMBED_DIM = 32  # Tamaño de las representaciones semánticas
NUM_EXPERTS = 3  # Número inicial de expertos (puedes agregar más)

# ---------- Encoder Semántico ----------
class SemanticEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
    def forward(self, x):
        return self.net(x)

# ---------- Memoria tipo clave-valor ----------
class EpisodicMemory:
    def __init__(self):
        self.keys = []
        self.values = []

    def add(self, key, value):
        self.keys.append(key.detach())
        self.values.append(value)

    def retrieve(self, query, top_k=1):
        if not self.keys:
            return None
        dists = [torch.norm(query - k) for k in self.keys]
        idxs = sorted(range(len(dists)), key=lambda i: dists[i])[:top_k]
        return [self.values[i] for i in idxs]

# ---------- Router Neuronal (RL simple) ----------
class Router(nn.Module):
    def __init__(self, embed_dim, num_experts):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_experts)
    def forward(self, embedding):
        logits = self.linear(embedding)
        return F.softmax(logits, dim=-1)

# ---------- Expertos (simples y especializados) ----------
class ExpertSum(nn.Module):
    def forward(self, x):  # x = [a, b, ?]
        return torch.tensor([x[0] + x[1]])

class ExpertRand(nn.Module):
    def forward(self, x):
        return torch.tensor([random.randint(0, 10)])

class ExpertZero(nn.Module):
    def forward(self, x):
        return torch.tensor([0.0])

# ---------- Sistema completo ----------
class CognitiveMoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = SemanticEncoder(input_dim=3, embed_dim=EMBED_DIM)
        self.router = Router(embed_dim=EMBED_DIM, num_experts=NUM_EXPERTS)
        self.memory = EpisodicMemory()
        self.experts = nn.ModuleList([ExpertSum(), ExpertRand(), ExpertZero()])

    def forward(self, x):
        emb = self.encoder(x.float())
        probs = self.router(emb)
        choice = torch.multinomial(probs, 1).item()
        output = self.experts[choice](x)

        # Guardar experiencia
        self.memory.add(emb, choice)

        return output, choice

# ---------- Ejemplo de entorno simulado ----------
def environment(x):  # Input = [a, b, ?], output = reward si acierta
    target = x[0] + x[1]
    return target

# ---------- Ciclo de entrenamiento simulado ----------
model = CognitiveMoE()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(500):
    a, b = random.randint(0, 10), random.randint(0, 10)
    x = torch.tensor([a, b, 0])
    y_true = environment(x)

    y_pred, expert_id = model(x)
    reward = -abs(y_pred.item() - y_true)  # reward negativo = error

    # Reinforce loss para entrenar router
    emb = model.encoder(x.float())
    probs = model.router(emb)
    log_prob = torch.log(probs[expert_id] + 1e-8)
    loss = -log_prob * reward

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 50 == 0:
        print(f"Paso {step}: a={a}, b={b}, experto={expert_id}, pred={y_pred.item():.2f}, reward={reward:.2f}")

##Modelo Multimodal Bioinspirado: Imagen + Texto Sintético

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device to CPU
device = torch.device("cpu")

# ----------------------------
# 1. Generador de texto falso (embedding aleatorio por imagen)
# ----------------------------
def generate_fake_text_embedding(batch_size, embed_dim=50):
    return torch.randn(batch_size, embed_dim, device=device)

# ----------------------------
# 2. Router Tálamo-Cortical
# ----------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ----------------------------
# 3. Expertos
# ----------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        q, k, v = self.query(x).unsqueeze(1), self.key(x).unsqueeze(1), self.value(x).unsqueeze(1)
        attn_weights = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        context = torch.bmm(attn_weights, v).squeeze(1)
        return self.fc(context)

# ----------------------------
# 4. Mezcla Cortical Multimodal
# ----------------------------
class CorticalMoE(nn.Module):
    def __init__(self, image_dim, text_dim):
        super().__init__()
        self.input_dim = image_dim + text_dim
        self.router = Router(self.input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(self.input_dim, 128),
            LogicExpert(self.input_dim),
            InstinctExpert(self.input_dim),
            ContextExpert(self.input_dim)
        ])

    def forward(self, image, text_embed):
        x = torch.cat((image, text_embed), dim=1)
        expert_idx, log_prob, probs = self.router(x)
        outputs = torch.stack([expert(x) for expert in self.experts])
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

# ----------------------------
# 5. Entrenamiento Multimodal
# ----------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch, text_embed_dim=50):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4, device=device)

    for images, labels in dataloader:
        images = images.view(images.size(0), -1).to(device)
        labels = labels.to(device)
        text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

        optimizer.zero_grad()
        router_optimizer.zero_grad()

        output, log_prob, expert_idx, _ = model(images, text_embeds)
        classification_loss = F.cross_entropy(output, labels)
        reward = -classification_loss.detach()
        reinforce_loss = -log_prob.mean() * reward
        loss = classification_loss + reinforce_loss

        loss.backward()
        optimizer.step()
        router_optimizer.step()

        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Época {epoch} | Pérdida: {total_loss/len(dataloader):.4f} | Uso por experto (entrenamiento): {expert_usage.tolist()}")

# ----------------------------
# 6. Evaluación (Test)
# ----------------------------
def test_loop(model, dataloader, text_embed_dim=50):
    model.eval()
    correct = 0
    total = 0
    expert_usage = torch.zeros(4, device=device)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

            output, _, expert_idx, _ = model(images, text_embeds)
            _, predicted = torch.max(output.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            for i in expert_idx:
                expert_usage[i] += 1

    accuracy = 100 * correct / total
    print(f'Precisión del modelo en el conjunto de prueba: {accuracy:.2f}%')
    print(f"Uso por experto (prueba): {expert_usage.tolist()}")
    return accuracy, expert_usage

# ----------------------------
# 7. Main
# ----------------------------
def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform) # Test set
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False) # Test loader

    image_dim = 28*28
    text_dim = 50

    model = CorticalMoE(image_dim, text_dim).to(device)
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    optimizer = torch.optim.Adam(expert_params, lr=1e-3)
    router_optimizer = torch.optim.Adam(router_params, lr=1e-3)

    for epoch in range(1, 6):
        train_loop(model, train_loader, optimizer, router_optimizer, epoch, text_embed_dim=text_dim)

    # Evaluate the model after training
    accuracy, expert_usage = test_loop(model, test_loader, text_embed_dim=text_dim)

    # Visualize expert usage
    expert_names = ['Memoria', 'Lógica', 'Instinto', 'Contexto']
    expert_usage_percentage = (expert_usage / expert_usage.sum() * 100).tolist()

    plt.figure(figsize=(8, 6))
    plt.bar(expert_names, expert_usage_percentage, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
    plt.xlabel('Expertos')
    plt.ylabel('Porcentaje de Uso')
    plt.title('Uso de Expertos en el Conjunto de Prueba')
    plt.ylim(0, 100)
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.savefig('expert_usage.png')

if __name__ == "__main__":
    main()

##Aquí tienes un modelo completo estilo "cortical-thalamic" con:

Router RL que decide qué experto (bioma) usar por entrada.

Cuatro expertos especializados: Memoria (LSTM), Cálculo (MLP profundo), Instinto (shallow), Contexto (autoatención).

Entrenamiento tipo REINFORCE para el router, que aprende a seleccionar expertos útiles.


Incluye un esquema de uso de expertos para visualizar si hay colapso (todos eligen el mismo) y se puede extender fácilmente con mecanismos de entropía o penalización si lo deseas.

¿Quieres que le incorpore feedback de salida al router (memoria interbioma), o implementamos la regularización del router ahora?

##Pensaste sumamente bien, te lo agradezco, si me gustaría cada esperto es una región al final no perdemos la idea lo que es bueno sigua así como estabas 🤔

1. Biomas especializados:



Primer esperto Memoria → con LSTM

Segundo Cálculo → MLP profundo

Tercero Instinto → red shallow

Cuarto: Contexto → con atención

2. Regularización del router:



Penalizar uso excesivo de un solo bioma.

Maximizar entropía para forzar especialización.

3. Memory feedback loop:



Mandar la salida de un bioma al router como hint en el siguiente paso (memoria interbioma).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ThalamusBiomeNet(nn.Module):
    def __init__(self, input_dim, shared_dim, biome_dim, output_dim, k_active=2):
        super().__init__()

        self.k_active = k_active
        self.num_biomes = 4

        # Feature extractor (sensorial cortex)
        self.feature_encoder = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU()
        )

        # Router (thalamus)
        self.thalamus_router = nn.Linear(shared_dim + output_dim, self.num_biomes)

        # Biome 1: Memory (LSTM)
        self.memory_lstm = nn.LSTM(shared_dim, biome_dim, batch_first=True)
        self.memory_proj = nn.Linear(biome_dim, output_dim)

        # Biome 2: Deep MLP (Calculation)
        self.calculation_mlp = nn.Sequential(
            nn.Linear(shared_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, output_dim)
        )

        # Biome 3: Instinct (Shallow)
        self.instinct_net = nn.Sequential(
            nn.Linear(shared_dim, 64), nn.ReLU(),
            nn.Linear(64, output_dim)
        )

        # Biome 4: Context (Self-Attention)
        self.context_attention = nn.MultiheadAttention(embed_dim=shared_dim, num_heads=4, batch_first=True)
        self.context_proj = nn.Linear(shared_dim, output_dim)

        # Initial feedback (output from previous step, zeroed initially)
        self.register_buffer("last_output", torch.zeros(1, output_dim))

    def forward(self, x):
        batch_size = x.size(0)
        shared_feat = self.feature_encoder(x)  # [B, shared_dim]

        # Repeat feedback memory to match batch size
        feedback = self.last_output.expand(batch_size, -1)

        # Routing decision
        router_input = torch.cat([shared_feat, feedback], dim=1)
        routing_logits = self.thalamus_router(router_input)

        # Top-k routing
        topk_vals, topk_idx = torch.topk(routing_logits, self.k_active, dim=1)
        routing_mask = torch.zeros_like(routing_logits)
        routing_mask.scatter_(1, topk_idx, 1)
        routing_weights = F.softmax(topk_vals, dim=1)

        # Prepare routing weights
        routing_weights_exp = torch.zeros_like(routing_logits).unsqueeze(-1)
        routing_weights_exp.scatter_(1, topk_idx.unsqueeze(-1), routing_weights.unsqueeze(-1))

        # Compute each biome output
        outputs = []

        # Memory biome (LSTM) — requiere input de 3D: [B, T=1, D]
        memory_input = shared_feat.unsqueeze(1)
        memory_out, _ = self.memory_lstm(memory_input)
        memory_final = self.memory_proj(memory_out[:, -1, :])
        outputs.append(memory_final)

        # Calculation biome (MLP)
        outputs.append(self.calculation_mlp(shared_feat))

        # Instinct biome (Shallow net)
        outputs.append(self.instinct_net(shared_feat))

        # Context biome (Attention) — también input 3D
        context_input = shared_feat.unsqueeze(1)
        attn_output, _ = self.context_attention(context_input, context_input, context_input)
        context_final = self.context_proj(attn_output.squeeze(1))
        outputs.append(context_final)

        # Stack biome outputs: [B, num_biomes, output_dim]
        biome_outputs = torch.stack(outputs, dim=1)

        # Weighted sum
        final_output = torch.sum(biome_outputs * routing_weights_exp, dim=1)

        # Feedback loop update (detached to prevent gradient leak)
        self.last_output = final_output.detach().mean(dim=0, keepdim=True)

        # Regularization: entropy penalty (high entropy => diverse routing)
        routing_probs = F.softmax(routing_logits, dim=1)
        entropy = -torch.sum(routing_probs * torch.log(routing_probs + 1e-8), dim=1).mean()

        return final_output, entropy

##🧠📡 Diseño: Tálamo Cortical Modular (Router de Biomas)

🔬 Concepto funcional:

El input sensorial es procesado por un extractor de características (como la corteza primaria).

El tálamo cortical (router) decide cuáles biomas activar (de forma suave o discreta).

Cada bioma representa un sistema funcional: memoria, cálculo, razonamiento, instinto...

La salida es una mezcla ponderada o selectiva de esos biomas.

##1. Un "tronco" (corteza central) que decide dónde se procesa cada input.


2. Un conjunto de módulos expertos o biomas, cada uno especializado.


3. Un sistema de enrutamiento inteligente, tipo Mixture of Experts (MoE), inspirado en cómo el cerebro delega funciones.



Tu instinto va bien. Lo que faltó fue integrar el Mecanismo de Enrutamiento (Router) de forma explícita y aprendible.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CerebralRouterNet(nn.Module):
    def __init__(self, input_dim, shared_dim, num_experts, expert_dim, output_dim):
        super().__init__()

        # 1. Corteza sensorial (feature extractor básico)
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU()
        )

        # 2. Tronco central que decide (router)
        self.router = nn.Sequential(
            nn.Linear(shared_dim, num_experts),
            nn.Softmax(dim=1)  # Pondera a qué experto enviar el input
        )

        # 3. Biomas especializados (expertos)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(shared_dim, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, output_dim)
            ) for _ in range(num_experts)
        ])

    def forward(self, x):
        shared_feat = self.shared_encoder(x)
        routing_weights = self.router(shared_feat)  # Tamaño: [batch, num_experts]

        # Procesar con todos los expertos en paralelo
        expert_outputs = torch.stack([expert(shared_feat) for expert in self.experts], dim=1)  # [batch, num_experts, output_dim]

        # Mezclar salidas de expertos según pesos del router
        routing_weights = routing_weights.unsqueeze(-1)  # [batch, num_experts, 1]
        final_output = torch.sum(expert_outputs * routing_weights, dim=1)  # [batch, output_dim]

        return final_output

#Router

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import numpy as np

# --- Config ---
config = {
    "input_dim": 28*28,
    "batch_size": 64,
    "lr": 1e-3,
    "epochs": 5,
    "model_path": "/content/cortical_moe_best.pth"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Router y Expertos ---
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        idx = dist.sample()
        logp = dist.log_prob(idx)
        return idx, logp, probs

class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1) # Add time dimension for LSTM
        _, (h, _) = self.lstm(x) # Get hidden state
        return self.fc(h.squeeze(0)) # Use hidden state for classification


class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.q = nn.Linear(input_dim, 64)
        self.k = nn.Linear(input_dim, 64)
        self.v = nn.Linear(input_dim, 64)
        self.out = nn.Linear(64, 10)

    def forward(self, x):
        # Attention expects a sequence dimension, but the input x is [B, input_dim].
        # Unsqueeze to simulate a sequence of length 1 if needed for self-attention,
        # but a single step attention is just a linear combination.
        # The logic below performs a simple self-attention like mechanism on the single input vector.
        q = self.q(x).unsqueeze(1) # [B, 1, 64]
        k = self.k(x).unsqueeze(1) # [B, 1, 64]
        v = self.v(x).unsqueeze(1) # [B, 1, 64]

        attn = torch.bmm(q, k.transpose(1,2)) # [B, 1, 1]
        attn = F.softmax(attn, dim=-1)      # [B, 1, 1]

        context = torch.bmm(attn, v).squeeze(1) # [B, 1, 64] -> [B, 64]
        return self.out(context)


class CorticalMoE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.router = Router(input_dim, 4)
        # Ensure experts are correctly instantiated with parameters defined in their __init__
        self.experts = nn.ModuleList([
            MemoryExpert(input_dim, 128), # Pass hidden_dim
            LogicExpert(input_dim),
            InstinctExpert(input_dim),
            ContextExpert(input_dim)
        ])
    def forward(self, x):
        idx, logp, probs = self.router(x)
        # Ensure expert outputs are collected and the correct one is selected per batch item
        # This assumes batch size > 1 and each item goes to potentially a different expert
        outs = torch.stack([expert(x) for expert in self.experts])  # [num_experts, B, 10]
        # Select the output based on the sampled index for each item in the batch
        # Use advanced indexing: outs[idx, batch_indices]
        batch_indices = torch.arange(x.size(0)).to(x.device) # Ensure indices are on the same device
        selected_out = outs[idx, batch_indices]

        return selected_out, logp, idx, probs

    def predict_with_expert(self, x, expert_idx):
        # Add batch dimension if input is a single sample
        if x.ndim == 1:
             x = x.unsqueeze(0)
        return self.experts[expert_idx](x)


def train_loop(model, loader, opt, r_opt):
    model.train()
    usage = torch.zeros(4).to(device)
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        r_opt.zero_grad() # Zero gradients for router optimizer

        # out, logp, idx, _ = model(x) # router_probs is not used here
        out, logp, idx, router_probs = model(x) # Capture router_probs if needed later, though not used for loss

        # Classification Loss
        loss_cls = F.cross_entropy(out, y)

        # Reinforcement Learning component for the Router
        # Reward is based on the classification loss (negative loss)
        # Penalize the router for choosing an expert that performed poorly
        # Adding 1.0 to make reward positive or less negative
        reward = -loss_cls.detach() # Use detach() so the loss gradient doesn't flow into the reward calculation

        # Reinforce the action (expert choice) based on the reward
        # The log probability of the chosen action is multiplied by the reward
        # Since we want to maximize reward, we minimize -logp * reward
        reinforce = -logp.mean() * reward # Use mean() of logp across the batch

        # Total loss is the sum of classification loss and the router's reinforcement loss
        total_loss_batch = loss_cls + reinforce

        # Backpropagation
        total_loss_batch.backward()

        # Update parameters
        opt.step() # Update expert parameters
        r_opt.step() # Update router parameters

        # Track expert usage
        for i in idx: usage[i] += 1
        total_loss += loss_cls.item() # Track only classification loss for reporting

    # Return average classification loss and expert usage
    return total_loss / len(loader), usage.tolist()


def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)

    model = CorticalMoE(config["input_dim"]).to(device)

    # Correctly separate router and expert parameters
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    # Ensure expert_params is not empty
    if not list(expert_params): # Convert to list to check if empty
         raise ValueError("Expert parameters list is empty. Check model layer definitions.")


    opt = optim.Adam(expert_params, lr=config["lr"])
    r_opt = optim.Adam(router_params, lr=config["lr"])

    best_acc, best_state = 0, None
    for epoch in range(1, config["epochs"] + 1):
        loss, usage = train_loop(model, loader, opt, r_opt)
        print(f"Ep {epoch} | Loss {loss:.4f} | Uso expertos: {usage}")

    torch.save(model.state_dict(), config["model_path"])
    print("Modelo guardado:", config["model_path"])

main()

In [ ]:
# Asume que ya ejecutaste Celda 1 antes

def evaluate(model, loader):
    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out, _, _, _ = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
    print(f"Precisión en test: {100 * correct / total:.2f}%")

def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    loader = DataLoader(test_data, batch_size=64)

    model = CorticalMoE(config["input_dim"]).to(device)
    model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
    print("Modelo cargado desde:", config["model_path"])
    evaluate(model, loader)

main()

#ReroutingNetwork

In [ ]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import trange
import os

# -------- Config --------
config = {
    "state_dim": 5,
    "num_actions": 2,
    "hidden_dim": 128,
    "lr": 0.01,
    "episodes": 500,
    "steps_per_episode": 20,
    "model_path": "/content/routing_rl_best_model.pt"
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- Modelo --------
class ReroutingNetwork(nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )
    def forward(self, x): return self.net(x)

# -------- Entorno simple --------
class SimpleEnv:
    def __init__(self):
        self.s = config["state_dim"]
        self.a = config["num_actions"]
        self.cur = np.random.randint(self.s)
        self.rewards = {(0,0):1, (1,0):2, (2,1):1, (3,1):3, (4,1):5}
        self.trans = {(0,0):1, (1,0):2, (2,1):3, (3,1):0, (4,1):1}
    def reset(self):
        self.cur = np.random.randint(self.s)
        s = np.eye(self.s)[self.cur]
        return torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)
    def step(self, a):
        r = self.rewards.get((self.cur, a), 0)
        self.cur = self.trans.get((self.cur, a), self.cur)
        s = np.eye(self.s)[self.cur]
        return torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0), r, False

# -------- Entrenamiento --------
def train():
    env = SimpleEnv()
    model = ReroutingNetwork(config["state_dim"], config["num_actions"], config["hidden_dim"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    best_reward = -float("inf")

    for ep in trange(config["episodes"], desc="Entrenando RL"):
        state = env.reset()
        log_probs, rewards = [], []

        for _ in range(config["steps_per_episode"]):
            logits = model(state)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            next_state, reward, _ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        G = sum(rewards)
        loss = -torch.stack([lp * G for lp in log_probs]).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if G > best_reward:
            best_reward = G
            torch.save(model.state_dict(), config["model_path"])

    print(f"\nMejor modelo guardado en: {config['model_path']}")

if __name__ == "__main__":
    train()

In [ ]:
# load_model.py
import torch
import os
import numpy as np

config = {
    "state_dim": 5,
    "num_actions": 2,
    "hidden_dim": 128,
    "model_path": "/content/routing_rl_best_model.pt"
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ReroutingNetwork(torch.nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=128):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, num_actions)
        )
    def forward(self, x): return self.net(x)

def run():
    if not os.path.exists(config["model_path"]):
        raise FileNotFoundError("Modelo no encontrado")

    model = ReroutingNetwork(config["state_dim"], config["num_actions"], config["hidden_dim"]).to(device)
    model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
    model.eval()

    # Estado de prueba
    state = np.zeros(config["state_dim"])
    state[0] = 1  # ejemplo: estado inicial 0
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(state)
        action = torch.argmax(logits, dim=1).item()
        print(f"Acción sugerida desde el estado 0: {action}")

if __name__ == "__main__":
    run()